# Video Downloader

In [60]:
import subprocess
import os
import yt_dlp
import math

def _converter_tempo_para_segundos(t):
    """
    Converte 't' para segundos.
    - str com ':' → aceita SS, MM:SS, HH:MM:SS
    - float com parte fracionária → interpreta como MM.SS (ex: 1.43 => 1 minuto 43 segundos)
    - int → interpreta como SEGUNDOS (ex: 8 => 8s)
    """
    # strings com ':'
    if isinstance(t, str):
        partes = t.split(':')
        partes = [p.strip() for p in partes if p != '']
        partes = list(reversed(partes))  # facilita: [ss, mm, hh]
        segundos = 0
        try:
            if len(partes) >= 1:
                segundos += int(partes[0])              # segundos
            if len(partes) >= 2:
                segundos += int(partes[1]) * 60         # minutos
            if len(partes) >= 3:
                segundos += int(partes[2]) * 3600       # horas
        except ValueError:
            raise ValueError(f"Formato de tempo inválido: {t}")
        return segundos

    # floats com parte fracionária → interpretar como MM.SS (minuto.segundos)
    if isinstance(t, float):
        minutos = int(math.floor(t))
        frac = t - minutos
        # transformamos a parte fracionária em "segundos" assumindo que foi escrita como .SS → multiplica por 100
        segundos_da_parte_decimal = int(round(frac * 100))
        if segundos_da_parte_decimal >= 60:
            # se a parte decimal ≥ 60, provavelmente o usuário quis minutos decimais (ex: 1.75 = 1.75 minutos)
            # nesse caso convertemos minutos decimais para segundos:
            return int(round(t * 60))
        return minutos * 60 + segundos_da_parte_decimal

    # inteiros → tratamos como SEGUNDOS (mais natural para 8, 90, etc.)
    if isinstance(t, int):
        return int(t)

    raise TypeError("Tempo deve ser string (MM:SS), float (MM.SS) ou int (segundos).")

def baixar_trecho_video(url, inicio, fim, saida="video_trecho.mp4"):
    """
    Baixa um vídeo em 720p (se disponível) e salva apenas o trecho desejado.
    'inicio' e 'fim' podem ser:
      - "0:08", "1:43" (string com :)
      - 0.08, 1.43 (float no formato MM.SS)
      - 8, 103 (int em segundos)
    """
    temp = "temp_download.mp4"

    opcoes_yt = {
        'format': 'bestvideo[height<=720]+bestaudio/best[height<=720]',
        'outtmpl': temp,
        'merge_output_format': 'mp4',
        'quiet': False,
    }

    print("⬇️  Baixando vídeo...")
    with yt_dlp.YoutubeDL(opcoes_yt) as ydl:
        ydl.download([url])

    print("🕐 Calculando tempos...")
    inicio_seg = _converter_tempo_para_segundos(inicio)
    fim_seg = _converter_tempo_para_segundos(fim)

    if fim_seg <= inicio_seg:
        raise ValueError(f"O tempo final ({fim}) deve ser maior que o inicial ({inicio}).")

    duracao_seg = fim_seg - inicio_seg

    print(f"✂️  Cortando trecho: início={inicio_seg}s, duração={duracao_seg}s ...")

    subprocess.run([
        'ffmpeg', '-y', '-i', temp,
        '-ss', str(inicio_seg), '-t', str(duracao_seg),
        '-c', 'copy', saida
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

    os.remove(temp)
    print(f"✅ Trecho salvo em: {saida}")


def baixar_varios_videos(videos):
    """
    Recebe uma lista de vídeos (url, início, fim, saída)
    e baixa todos com barra de progresso global.
    """
    total = len(videos)
    print(f"📦 Baixando {total} vídeo(s)...\n")

    for i, v in enumerate(videos, start=1):
        print(f"📹 [{i}/{total}] Processando {v['saida']}...")
        baixar_trecho_video(v['url'], inicio=v['inicio'], fim=v['fim'], saida=v['saida'])


## Creating videos path

In [55]:
!mkdir videos2

In [56]:
VIDEOS_PATH = "./videos2/"

## Defining Videos

In [61]:
videos = [
    {
        # Electric Callboy - Everytime We Touch - [MUSIC VIDEO]
        "url": "https://www.youtube.com/watch?v=AuBXeF5acqE",
        "inicio": 0.05,   # minuto inicial: 0:05
        "fim": 2.05,      # minuto final:   2:05
        "saida": VIDEOS_PATH+"music1.mp4"
    },
    {
        # Crazy Frog - Axel F  - [MUSIC VIDEO]
        "url": "https://www.youtube.com/watch?v=k85mRPqvMbE",
        "inicio": 0.00,   # minuto inicial: 0:00
        "fim": 2.00,      # minuto final:   2:00
        "saida": VIDEOS_PATH+"music2.mp4"
    },
    
    {
        # Mickey 17 | Official Trailer - [MOVIE TRAILER]
        "url": "https://www.youtube.com/watch?v=osYpGSz_0i4",
        "inicio": 0.00,   # minuto inicial: 0:00 
        "fim": 2.00,      # minuto final:   2:00
        "saida": VIDEOS_PATH+"movie1.mp4"
    },
    {
        # Tubarão (1975) - [MOVIE TRAILER]
        "url": "https://www.youtube.com/watch?v=CxLG9BMs5yY",
        "inicio": 0.00,   # minuto inicial: 0:00 
        "fim": 0.54,      # minuto final:   0:54
        "saida": VIDEOS_PATH+"movie2.mp4"
    },

    {
        # Fullmetal Alchemist: Brotherhood | Trailer - [ANIME]
        "url": "https://www.youtube.com/watch?v=kx0nBaS_q50",
        "inicio": 0.00,   # minuto inicial: 0:00 
        "fim": 1.15,      # minuto final:   1:15
        "saida": VIDEOS_PATH+"anime1.mp4"
    },
    {
        # YOUR NAME ( Kimi no nawa )- [ANIME]
        "url": "https://www.youtube.com/watch?v=E5J6Vmcr1j4",
        "inicio": 0.00,   # minuto inicial: 0:00 
        "fim": 1.15,      # minuto final:   1:15
        "saida": VIDEOS_PATH+"anime2.mp4"
    },
]

## Downloading

In [63]:
baixar_varios_videos(videos)

📦 Baixando 6 vídeo(s)...

📹 [1/6] Processando ./videos2/music1.mp4...
⬇️  Baixando vídeo...
[youtube] Extracting URL: https://www.youtube.com/watch?v=AuBXeF5acqE
[youtube] AuBXeF5acqE: Downloading webpage
[youtube] AuBXeF5acqE: Downloading tv client config
[youtube] AuBXeF5acqE: Downloading tv player API JSON
[youtube] AuBXeF5acqE: Downloading web safari player API JSON
[youtube] AuBXeF5acqE: Downloading m3u8 information
[info] AuBXeF5acqE: Downloading 1 format(s): 398+251
[download] Sleeping 4.00 seconds as required by the site...
[download] Destination: temp_download.f398.mp4
[download] 100% of   16.21MiB in 00:00:00 at 496.13MiB/s   
[download] Destination: temp_download.f251.webm
[download] 100% of    3.39MiB in 00:00:00 at 42.85MiB/s  
[Merger] Merging formats into "temp_download.mp4"
Deleting original file temp_download.f251.webm (pass -k to keep)
Deleting original file temp_download.f398.mp4 (pass -k to keep)
🕐 Calculando tempos...
✂️  Cortando trecho: início=5s, duração=120s ..

[youtube] k85mRPqvMbE: Downloading m3u8 information
[info] k85mRPqvMbE: Downloading 1 format(s): 95
[download] Sleeping 5.00 seconds as required by the site...
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 35
[download] Destination: temp_download.mp4
[download] 100% of   24.52MiB in 00:00:01 at 14.10MiB/s                  
[FixupM3u8] Fixing MPEG-TS in MP4 container of "temp_download.mp4"
🕐 Calculando tempos...
✂️  Cortando trecho: início=0s, duração=120s ...
✅ Trecho salvo em: ./videos2/music2.mp4
📹 [3/6] Processando ./videos2/movie1.mp4...
⬇️  Baixando vídeo...
[youtube] Extracting URL: https://www.youtube.com/watch?v=osYpGSz_0i4
[youtube] osYpGSz_0i4: Downloading webpage
[youtube] osYpGSz_0i4: Downloading tv client config
[youtube] osYpGSz_0i4: Downloading tv player API JSON
[youtube] osYpGSz_0i4: Downloading web safari player API JSON
[youtube] osYpGSz_0i4: Downloading m3u8 information
[info] osYpGSz_0i4: Downloading 1 format(s): 398+251
[download] Sleeping 4.0

# Cut Detection

In [70]:
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

VIDEOS_PATH = "./videos2/converted/"
CUTS_CSV = "./cuts.csv"  # arquivo enviado por você

## Funções Auxiliares

In [16]:
def list_videos(videos_path=VIDEOS_PATH, exts=("mp4","avi","mov","mkv")):
    files = []
    for e in exts:
        files.extend(glob(os.path.join(videos_path, f"**/*.{e}"), recursive=True))
    files.sort()
    return files

In [40]:
def read_manual_cuts(csv_path=CUTS_CSV):
    """
    Lê o cuts.csv no formato:
      video_id, frame, time_start_s, time_end_s, ...
    Retorna dict {video_base_name: np.array([frames])}
    e adiciona variantes com sufixo '_h264' para compatibilidade.
    """
    import os
    import pandas as pd
    df = pd.read_csv(csv_path)
    if 'video_id' not in df.columns or 'frame' not in df.columns:
        raise ValueError("CSV precisa ter colunas 'video_id' e 'frame'")
    
    cuts = {}
    for vid, g in df.groupby('video_id'):
        frames = np.array(sorted(g['frame'].astype(int).values))
        cuts[vid] = frames
        # também adiciona versões com sufixo compatível com vídeos convertidos
        cuts[vid + "_h264"] = frames
        cuts[vid + "_h264.mp4"] = frames
        cuts[vid + ".mp4"] = frames
    return cuts


## Histograma Local

In [20]:
def local_histogram(frame, partitions=(2,2), bins=16, channel='rgb'):
    """
    Calcula histograma local por partição.
    frame: HxWx3 uint8
    partitions: (ph, pw) número de partições (linhas, colunas)
    bins: número de bins por canal
    channel: 'rgb' ou 'y' (se quiser usar luminância)
    Retorna vetor 1D normalizado (concatenação dos histogramas de cada bloco e canal).
    """
    if frame.ndim != 3 or frame.shape[2] != 3:
        raise ValueError("frame deve ser HxWx3")
    h, w, _ = frame.shape
    ph, pw = partitions
    hist_blocks = []
    # escolher canais
    if channel == 'y':
        # converter para luminância aproximada
        f = (0.299*frame[:,:,2] + 0.587*frame[:,:,1] + 0.114*frame[:,:,0]).astype(np.uint8)
        channels = [f]
    else:
        # frame em BGR (cv2) -> transformar para RGB para coerência
        # Se frame já está em RGB, mantenha. Nosso frame virá de cv2.imread -> BGR.
        # Assumimos frame atual no formato BGR (padrão cv2). Para hist calculamos por canais B,G,R.
        channels = [frame[:,:,0], frame[:,:,1], frame[:,:,2]]
    for i in range(ph):
        r0 = int(round(i * h / ph))
        r1 = int(round((i+1) * h / ph))
        for j in range(pw):
            c0 = int(round(j * w / pw))
            c1 = int(round((j+1) * w / pw))
            for ch in channels:
                blk = ch[r0:r1, c0:c1].ravel()
                if blk.size == 0:
                    hist = np.zeros(bins, dtype=float)
                else:
                    hist, _ = np.histogram(blk, bins=bins, range=(0,256))
                    s = hist.sum()
                    if s > 0:
                        hist = hist.astype(float) / s
                    else:
                        hist = hist.astype(float)
                hist_blocks.append(hist)
    return np.concatenate(hist_blocks)  # vetor 1D


In [21]:
def hist_diff(h1, h2, metric='l1'):
    """
    Diferença entre vetores de histograma (já normalizados).
    metric: 'l1' (default) ou 'chi2' (opcional)
    """
    if metric == 'l1':
        return np.sum(np.abs(h1 - h2))
    elif metric == 'chi2':
        # somar 0.5*(a-b)^2/(a+b+eps)
        eps = 1e-10
        num = (h1 - h2) ** 2
        den = h1 + h2 + eps
        return 0.5 * np.sum(num / den)
    else:
        raise ValueError("metric inválido")


## BIC Local

In [22]:
def bic_local(frame, partitions=(2,2), bins=16, edge_threshold=20):
    """
    Implementação do BIC local:
    - Para cada partição, classifica pixels como 'border' ou 'interior'
      usando diferença média para vizinhos (4-neighborhood) e threshold.
    - Calcula histograma separado para border e interior (por canal)
    - Concatena tudo.
    frame: HxWx3 uint8 (BGR)
    Retorna vetor 1D
    """
    # converter para int para evitar underflow em subtrações
    f = frame.astype(np.int16)
    h, w, _ = f.shape
    ph, pw = partitions
    hist_blocks = []
    # cálculo de diferença média para vizinhos (4-neighbors)
    # compute absolute diff to up/down/left/right
    # pad to keep same shape
    # We'll compute a 'edge score' = mean absolute diff to neighbors across channels
    # vectorized:
    pad = np.pad(f, ((1,1),(1,1),(0,0)), mode='edge')
    center = pad[1:-1,1:-1,:]
    up = pad[0:-2,1:-1,:]
    down = pad[2:,1:-1,:]
    left = pad[1:-1,0:-2,:]
    right = pad[1:-1,2:,:]
    # absolute difference per neighbor and channel, then mean across neighbors and channels
    # shape HxW
    diff = (np.abs(center - up).sum(axis=2) +
            np.abs(center - down).sum(axis=2) +
            np.abs(center - left).sum(axis=2) +
            np.abs(center - right).sum(axis=2)) /  (4.0 * 3.0)  # média
    # border mask
    border_mask = diff > edge_threshold  # True = border pixel, False = interior
    for i in range(ph):
        r0 = int(round(i * h / ph))
        r1 = int(round((i+1) * h / ph))
        for j in range(pw):
            c0 = int(round(j * w / pw))
            c1 = int(round((j+1) * w / pw))
            blk = f[r0:r1, c0:c1, :].astype(np.uint8)
            mask_blk = border_mask[r0:r1, c0:c1]
            # pixels border and interior
            border_pixels = blk[mask_blk]
            interior_pixels = blk[~mask_blk]
            # para cada grupo e canal calcular histograma (bins)
            for group in (border_pixels, interior_pixels):
                if group.size == 0:
                    # adicionar zeros para cada canal
                    for ch in range(3):
                        hist_blocks.append(np.zeros(bins, dtype=float))
                else:
                    # group shape Nx3
                    for ch in range(3):
                        pix = group[:, ch]
                        hist, _ = np.histogram(pix, bins=bins, range=(0,256))
                        s = hist.sum()
                        if s > 0:
                            hist_blocks.append(hist.astype(float) / s)
                        else:
                            hist_blocks.append(hist.astype(float))
    return np.concatenate(hist_blocks)


In [23]:
def bic_diff(b1, b2, metric='l1'):
    return hist_diff(b1, b2, metric=metric)
    

## Leitura de Frames com salto

In [24]:
def iterate_video_frames(video_path, skip_frames=1, as_rgb=False, max_frames=None):
    """
    Itera frames do vídeo usando cv2.VideoCapture.
    skip_frames: pula N-1 frames entre frames retornados (se skip_frames=30 -> 1 frame a cada 30)
    Retorna (frame_index, frame) onde frame é HxWx3 uint8 (BGR)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Não foi possível abrir {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    idx = 0
    yielded = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % skip_frames == 0:
            if as_rgb:
                yield idx, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            else:
                yield idx, frame
            yielded += 1
            if max_frames and yielded >= max_frames:
                break
        idx += 1
    cap.release()

## Detector de Cortes

In [27]:
def detect_shots_from_video(video_path, descriptor_fn, diff_fn, partitions=(2,2),
                            bins=16, edge_threshold=20, skip_seconds_approx=1.0,
                            threshold=None, metric='l1', debug=False):
    """
    Detecta cortes num video aplicando:
      - lê 1 frame a cada skip_seconds_approx (aprox 1s)
      - calcula descritor para cada frame (descriptor_fn)
      - aplica diff_fn entre descritores consecutivos
      - considera corte quando diff > threshold
    descriptor_fn: função(frame, partitions, bins, edge_threshold) -> vetor
    diff_fn: função(d1, d2) -> número
    threshold: se None -> será estimado automaticamente (média + k*std)
    Retorna:
      - cuts_frames: lista de índices de frames onde ocorreu corte (índice do frame lido)
      - diffs: lista de diffs calculadas (na mesma ordem)
      - fps: fps do video
    """
    cap_test = cv2.VideoCapture(video_path)
    fps = cap_test.get(cv2.CAP_PROP_FPS) or 30.0
    cap_test.release()
    skip_frames = max(1, int(round(skip_seconds_approx * fps)))
    descriptors = []
    frame_indices = []
    # iterar e calcular descritores
    for idx, frame in iterate_video_frames(video_path, skip_frames=skip_frames, as_rgb=False):
        desc = descriptor_fn(frame, partitions=partitions, bins=bins, edge_threshold=edge_threshold)
        descriptors.append(desc)
        frame_indices.append(idx)
    descriptors = np.array(descriptors, dtype=object)  # array de objetos (vetores)
    # calcular diffs
    diffs = []
    for k in range(1, len(descriptors)):
        d = diff_fn(descriptors[k-1], descriptors[k])
        diffs.append(d)
    diffs = np.array(diffs)
    # threshold automático se for None
    if threshold is None:
        mu = diffs.mean() if diffs.size>0 else 0.0
        sigma = diffs.std() if diffs.size>0 else 0.0
        # escolha empírica: mu + 1.5*sigma (pode ajustar)
        threshold = mu + 1.5 * sigma
        if debug:
            print(f"[auto-threshold] mean={mu:.4f}, std={sigma:.4f}, threshold={threshold:.4f}")
    # detectar cortes: quando diffs[i] > threshold -> corte no frame_indices[i+1]
    cuts = []
    for i, d in enumerate(diffs):
        if d > threshold:
            # escolher o frame do segundo frame (i+1) como ponto do corte
            cuts.append(frame_indices[i+1])
    return np.array(cuts, dtype=int), diffs, frame_indices, fps, threshold

## Detector de Keyframe

In [28]:
def select_keyframes_from_cuts(video_path, cuts_frames, fps, strategy='center', window_radius=None):
    """
    Seleciona um keyframe por cena identificada (entre cortes).
    cuts_frames: array de frames detectados como cortes (frame index no vídeo)
    strategy:
      - 'center': pega o frame central entre previous_cut e next_cut
      - 'representative': dentro do segmento pega frame que minimiza soma das diferenças (mais representativo) - aqui simplificado como central
    window_radius: se quiser evitar usar frames próximo a cortes (em frames)
    Retorna dict {scene_index: (start_frame, end_frame, keyframe_index)}
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    # construir limites de cena
    cuts_sorted = np.sort(np.array(cuts_frames, dtype=int))
    starts = [0] + cuts_sorted.tolist()
    ends = cuts_sorted.tolist() + [total_frames-1]
    scenes = {}
    for i, (s,e) in enumerate(zip(starts, ends)):
        if strategy == 'center':
            k = (s + e) // 2
            # evitar muito próximo ao corte
            if window_radius:
                left = min(total_frames-1, s + window_radius)
                right = max(0, e - window_radius)
                k = min(max(k, left), right)
            scenes[i] = (s, e, int(k))
        else:
            # fallback para center
            k = (s + e) // 2
            scenes[i] = (s, e, int(k))
    return scenes

## Avaliação

In [30]:
def evaluate_detection(detected_cuts, manual_cuts, fps, tolerance_s=1.0):
    """
    Avalia acurácia (precisão simples) considerando tolerância em segundos.
    detected_cuts: array de frame indices detectados
    manual_cuts: array de frame indices manuais (do ground truth)
    fps: frames por segundo do vídeo
    Retorna:
      acuracia = acertos / N_manual
      acertos listados (parcial)
    Critério de acerto: existe detected d tal que |d - m| <= tol_frames
    """
    tol_frames = int(round(tolerance_s * fps))
    manual = np.array(manual_cuts, dtype=int)
    detected = np.array(detected_cuts, dtype=int)
    hits = 0
    matched_detected = set()
    matched_manual = set()
    for i, m in enumerate(manual):
        # buscar qualquer detected dentro da janela
        diffs = np.abs(detected - m)
        idxs = np.where(diffs <= tol_frames)[0]
        if idxs.size > 0:
            hits += 1
            # marcar o primeiro detectado como usado
            matched_detected.add(int(idxs[0]))
            matched_manual.add(i)
    acc = hits / len(manual) if len(manual) > 0 else 0.0
    return {
        'accuracy': acc,
        'hits': hits,
        'n_manual': len(manual),
        'matched_manual_indices': sorted(list(matched_manual)),
        'matched_detected_indices': sorted(list(matched_detected))
    }

In [72]:
def run_pipeline_all(videos_path=VIDEOS_PATH, cuts_csv=CUTS_CSV,
                     partitions=(2,2), bins=16, edge_threshold=20,
                     skip_seconds=1.0, threshold_hist=None, threshold_bic=None,
                     output_table_path="results_table.csv", debug=False):
    videos = list_videos(videos_path)
    manual = read_manual_cuts(cuts_csv)
    rows = []
    for vpath in tqdm(videos, desc="videos"):
        fname = os.path.basename(vpath)
        print("\n--- Processing:", fname)
        # HIST local
        cuts_hist, diffs_hist, frame_indices_hist, fps, thr_hist = detect_shots_from_video(
    vpath,
    descriptor_fn=lambda f, partitions, bins, **_: local_histogram(f, partitions=partitions, bins=bins),
    diff_fn=hist_diff,
    partitions=partitions, bins=bins, edge_threshold=edge_threshold,
    skip_seconds_approx=skip_seconds, threshold=threshold_hist, metric='l1', debug=debug
)
        # BIC local
        cuts_bic, diffs_bic, frame_indices_bic, fps_bic, thr_bic = detect_shots_from_video(
            vpath, descriptor_fn=lambda f, **kw: bic_local(f, **kw),
            diff_fn=bic_diff,
            partitions=partitions, bins=bins, edge_threshold=edge_threshold,
            skip_seconds_approx=skip_seconds, threshold=threshold_bic, metric='l1', debug=debug
        )
        # recuperar cortes manuais para este arquivo (se houver)
        manual_cuts = manual.get(fname, np.array([], dtype=int))
        # avaliar
        eval_hist = evaluate_detection(cuts_hist, manual_cuts, fps, tolerance_s=1.0)
        eval_bic = evaluate_detection(cuts_bic, manual_cuts, fps, tolerance_s=1.0)
        # selecionar keyframes (usando strategy center)
        kfs_hist = select_keyframes_from_cuts(vpath, cuts_hist, fps, strategy='center')
        kfs_bic = select_keyframes_from_cuts(vpath, cuts_bic, fps, strategy='center')
        # salvar resultados de linha
        rows.append({
            'filename': fname,
            'fps': fps,
            'manual_cuts': manual_cuts.tolist(),
            'cuts_hist': cuts_hist.tolist(),
            'cuts_bic': cuts_bic.tolist(),
            'threshold_hist': thr_hist,
            'threshold_bic': thr_bic,
            'accuracy_hist': eval_hist['accuracy'],
            'accuracy_bic': eval_bic['accuracy'],
            'keyframes_hist': {k:v[2] for k,v in kfs_hist.items()},
            'keyframes_bic': {k:v[2] for k,v in kfs_bic.items()}
        })
        # imprimir resumo rápido
        print(f"  manual: {manual_cuts}")
        print(f"  hist cuts: {cuts_hist}  acc={eval_hist['accuracy']:.3f}")
        print(f"  bic  cuts: {cuts_bic}  acc={eval_bic['accuracy']:.3f}")
    df = pd.DataFrame(rows)
    df.to_csv(output_table_path, index=False)
    print(f"\nResults saved to {output_table_path}")
    return df

# -------------------------
# 8) Execução de exemplo
# -------------------------
if __name__ == "__main__":
    # Parâmetros iniciais que você pode ajustar:
    partitions = (2,2)        # partitionamento mínimo 2x2
    bins = 16                 # bins por histograma
    edge_threshold = 20       # limiar de borda para BIC (ajustar)
    skip_seconds = 1.0        # pular ~1s entre frames (velociza)
    # thresholds None => calculados automaticamente
    results_df = run_pipeline_all(VIDEOS_PATH, CUTS_CSV,
                              partitions=(3,3), bins=16,
                              edge_threshold=15,
                              skip_seconds=0.5,   # mais amostragem
                              threshold_hist=None,
                              threshold_bic=None,
                              debug=False)

display(results_df)


videos:   0%|          | 0/6 [00:00<?, ?it/s]


--- Processing: anime1_h264.mp4
  manual: [  34   96  131  186  214  244  276  327  405  441  490  539  586  646
  686  723  768  783  815  886  980 1029 1081 1129 1184 1238 1275 1313
 1364 1414 1460 1552 1586 1622 1683 1723 1782 1827 1869 1957 2027]
  hist cuts: [ 105  225  285  330  825  900  945 1275 1560 1590 1635 1830 1875 1965
 2100]  acc=0.366
  bic  cuts: [ 105  225  285  660  735  825  900  945 1320 1560 1635 1695 1875 1965
 2100]  acc=0.439

--- Processing: anime2_h264.mp4
  manual: [ 133  243  354  621  774  840  909  967 1011 1131 1206 1235 1298 1366
 1429 1509 1560 1623]
  hist cuts: [  60  360  780  810  825  840  945  975 1020 1140 1215 1305 1440 1500
 1515 1560 1635]  acc=0.722
  bic  cuts: [  60  360  825  945  975 1020 1140 1215 1305 1440 1500 1515 1560 1635]  acc=0.667

--- Processing: movie1_h264.mp4
  manual: [  46  115  247  348  393  441  480  503  527  589  615  647  689  707
  732  757  787  809  828  900  942  956  975  999 1019 1066 1085 1118
 1150 1169 1200

,filename,fps,manual_cuts,cuts_hist,cuts_bic,threshold_hist,threshold_bic,accuracy_hist,accuracy_bic,keyframes_hist,keyframes_bic
0,anime1_h264.mp4,29.970030,"[34, 96, 131, 186, 214, 244, 276, 327, 405, 44...","[105, 225, 285, 330, 825, 900, 945, 1275, 1560...","[105, 225, 285, 660, 735, 825, 900, 945, 1320,...",39.593311,66.801242,0.365854,0.439024,"{0: 52, 1: 165, 2: 255, 3: 307, 4: 577, 5: 862...","{0: 52, 1: 165, 2: 255, 3: 472, 4: 697, 5: 780..."
1,anime2_h264.mp4,30.000000,"[133, 243, 354, 621, 774, 840, 909, 967, 1011,...","[60, 360, 780, 810, 825, 840, 945, 975, 1020, ...","[60, 360, 825, 945, 975, 1020, 1140, 1215, 130...",33.035483,56.033347,0.722222,0.666667,"{0: 30, 1: 210, 2: 570, 3: 795, 4: 817, 5: 832...","{0: 30, 1: 210, 2: 592, 3: 885, 4: 960, 5: 997..."
2,movie1_h264.mp4,23.976024,"[46, 115, 247, 348, 393, 441, 480, 503, 527, 5...","[396, 444, 528, 600, 648, 696, 900, 948, 960, ...","[396, 444, 528, 600, 648, 900, 948, 960, 1068,...",35.032359,64.584118,0.400000,0.360000,"{0: 198, 1: 420, 2: 486, 3: 564, 4: 624, 5: 67...","{0: 198, 1: 420, 2: 486, 3: 564, 4: 624, 5: 77..."
3,movie2_h264.mp4,30.000000,"[164, 175, 253, 319, 363, 380, 419, 431, 471, ...","[165, 180, 255, 375, 390, 405, 420, 540, 585, ...","[165, 180, 255, 375, 390, 420, 435, 525, 540, ...",25.296784,48.317215,0.875000,0.875000,"{0: 82, 1: 172, 2: 217, 3: 315, 4: 382, 5: 397...","{0: 82, 1: 172, 2: 217, 3: 315, 4: 382, 5: 405..."
4,music1_h264.mp4,25.000000,"[169, 773, 798, 860, 938, 1013, 1063, 1210, 12...","[1368, 1392, 1476, 1488, 1536, 1584, 1632, 164...","[972, 1116, 1368, 1392, 1476, 1632, 1644, 1668...",24.604934,54.185086,0.531915,0.489362,"{0: 684, 1: 1380, 2: 1434, 3: 1482, 4: 1512, 5...","{0: 486, 1: 1044, 2: 1242, 3: 1380, 4: 1434, 5..."
5,music2_h264.mp4,25.000000,"[93, 130, 181, 256, 299, 347, 402, 450, 523, 5...","[24, 36, 96, 168, 180, 300, 528, 1476, 1572, 1...","[24, 36, 96, 168, 180, 300, 528, 1476, 1524, 1...",33.471474,58.701739,0.276596,0.255319,"{0: 12, 1: 30, 2: 66, 3: 132, 4: 174, 5: 240, ...","{0: 12, 1: 30, 2: 66, 3: 132, 4: 174, 5: 240, ..."


In [67]:
import os, subprocess
from glob import glob

VIDEOS_PATH = "./videos2/"

def convert_to_h264(videos_path=VIDEOS_PATH):
    """Converte todos os vídeos para H.264 (caso não sejam lidos pelo OpenCV)."""
    files = []
    for ext in ("*.mp4", "*.mkv", "*.mov", "*.avi"):
        files.extend(glob(os.path.join(videos_path, ext)))
    if not files:
        print("Nenhum vídeo encontrado em", videos_path)
        return
    os.makedirs(os.path.join(videos_path, "converted"), exist_ok=True)
    for f in files:
        name = os.path.basename(f)
        out = os.path.join(videos_path, "converted", os.path.splitext(name)[0] + "_h264.mp4")
        if os.path.exists(out):
            print(f"✅ Já convertido: {out}")
            continue
        print(f"🎬 Convertendo {name} → {out}")
        cmd = [
            "ffmpeg", "-y", "-i", f,
            "-c:v", "libx264", "-crf", "20", "-preset", "medium",
            "-c:a", "aac", "-strict", "experimental", out
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        # testar se agora abre
        import cv2
        cap = cv2.VideoCapture(out)
        ret, frame = cap.read()
        cap.release()
        if ret:
            print(f"✅ Conversão OK: {name}")
        else:
            print(f"❌ Erro: {name} ainda não pôde ser lido.")
    print("\n✅ Todos os vídeos convertidos salvos em ./videos/converted/")

convert_to_h264(VIDEOS_PATH)


🎬 Convertendo movie2.mp4 → ./videos2/converted/movie2_h264.mp4
✅ Conversão OK: movie2.mp4
🎬 Convertendo anime1.mp4 → ./videos2/converted/anime1_h264.mp4
✅ Conversão OK: anime1.mp4
🎬 Convertendo anime2.mp4 → ./videos2/converted/anime2_h264.mp4
✅ Conversão OK: anime2.mp4
🎬 Convertendo music1.mp4 → ./videos2/converted/music1_h264.mp4
✅ Conversão OK: music1.mp4
🎬 Convertendo movie1.mp4 → ./videos2/converted/movie1_h264.mp4
✅ Conversão OK: movie1.mp4
🎬 Convertendo music2.mp4 → ./videos2/converted/music2_h264.mp4
✅ Conversão OK: music2.mp4

✅ Todos os vídeos convertidos salvos em ./videos/converted/


In [43]:
import pandas as pd

df = pd.read_csv("results_table.csv")
df["category"] = df["filename"].apply(lambda x: x.split("1")[0].split("2")[0])  # separa anime/movie/music
summary = df.groupby("category")[["accuracy_hist","accuracy_bic"]].mean()
print(summary)


          accuracy_hist  accuracy_bic
category                             
anime          0.544038      0.552846
movie          0.512500      0.461250
music          0.393617      0.404255


In [71]:
VIDEOS_PATH


'./videos2/converted/'